# Fan Impact Pathway 온톨로지(K→F→Persona) — 최종 산출물 적용 노트북

r22 노트북(`archive/v6_r22_era/fan_impact_ontology/fan_impact_pathway_ontology.ipynb`)은 K=8/M=6/실루엣 0.154 스냅샷으로 전략문서를
시연했다. 최종 제출본은 **동결 스냅샷 v7-40(7,350건, K=10 → M=5, 실루엣 0.267)**에서 F1~F5·페르소나를 확정했고, 라이브 10,020건
재적합(K=8/M=5/0.046)은 실루엣 게이트를 통과하지 못해 loyalty/spillover 점수만 라이브로 갱신했다(README 3층 구조).

| 입력 | 내용 |
|---|---|
| `lda_v6_diagnostics_frozen_v7_40.json` | 동결 진단(K-grid, K=10 상위어, topic_to_factor, factor 라벨) |
| `factor_pathway_map_v7.json` | raw Factor 라벨 → F1~F5 영향경로 매핑 + QA |
| `topic_cards_v7.json` | K0~K9 Topic Card(상위어·대표 문장·대표 팬덤) |
| `fan_persona_v7.json` | 팬덤별 Top-2 Factor 페르소나(43/31/17/9) + Factor-specific Impact |
| `persona_decision_space_v7.json` | PCA(분산비)·덴드로그램(컷 높이) |
| `fandom_scores_v6.json` | 동결 점수(factor_share 5개) |
| `lda_v6_diagnostics_live_reference_v7.json`, `lda_excluded_bullets_v7.json` | 라이브 재적합 진단(게이트 미통과)과 LDA 제외 불릿 2건 |

In [1]:
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 140)


def find_repo_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "data" / "v7_final" / "fandoms_v3_100.json").exists():
            return p
    raise FileNotFoundError("저장소 루트(data/v7_final/fandoms_v3_100.json)를 찾지 못함 — 저장소 안에서 실행하세요")


REPO = find_repo_root()
DATA_DIR = REPO / "data" / "v7_final"                       # 최종 산출물(10,020건 라이브 + 동결 스냅샷 7,350건)
ROUNDS_DIR = REPO / "data" / "v7_rounds"                    # 병합 로그 r1~r72
ARCHIVE_DIR = REPO / "archive" / "v6_r22_era" / "data" / "v6_r22_snapshot"   # r22(5,612건) 비교용, 읽기 전용


def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def flatten_bullets(fandoms):
    rows = []
    for rec in fandoms:
        for kind in ("loyalty", "spillover"):
            for item in rec.get(kind, []):
                rows.append({"fandom": rec["fandom"], "category": rec.get("category"),
                             "bullet_type": kind, "text": item.get("t", "") or "", "url": item.get("u", "") or ""})
    return pd.DataFrame(rows)

diag = load_json(DATA_DIR / "lda_v6_diagnostics_frozen_v7_40.json")
diag_live = load_json(DATA_DIR / "lda_v6_diagnostics_live_reference_v7.json")
pathway = load_json(DATA_DIR / "factor_pathway_map_v7.json")
cards = load_json(DATA_DIR / "topic_cards_v7.json")
persona = load_json(DATA_DIR / "fan_persona_v7.json")
space = load_json(DATA_DIR / "persona_decision_space_v7.json")
frozen_scores = load_json(DATA_DIR / "fandom_scores_v6.json")
excluded = load_json(DATA_DIR / "lda_excluded_bullets_v7.json")
scores_by = {d["fandom"]: d for d in frozen_scores}
print(f"동결 진단: K={diag['selected_k']}, M={diag['selected_m_meta_factors']}, silhouette={diag['meta_factor_silhouette']} | 근거문장 합 {sum(d['activity'] for d in frozen_scores)}")
print(f"라이브 재적합: K={diag_live['selected_k']}, M={diag_live['selected_m_meta_factors']}, silhouette={diag_live['meta_factor_silhouette']} (게이트 미통과)")
print(f"LDA 문서 수: {excluded['lda_document_count']} = {excluded['total_bullets']} - 제외 {excluded['excluded_count']}건(3토큰 미만)")

동결 진단: K=10, M=5, silhouette=0.267 | 근거문장 합 7350
라이브 재적합: K=8, M=5, silhouette=0.046 (게이트 미통과)
LDA 문서 수: 10018 = 10020 - 제외 2건(3토큰 미만)


## 1. 온톨로지 정의 — F Impact Pathway(전략보고서 3.1절) ↔ 최종 매핑 파일

In [2]:
F_IMPACT_PATHWAY = pd.DataFrame([
    {"F": "F1", "명칭": "팬덤결속 경로", "경로": "Fan → Fan", "대표신호": "팬클럽·기부·응원"},
    {"F": "F2", "명칭": "직접소비 경로", "경로": "Fan → Market", "대표신호": "앨범·티켓·굿즈·판매"},
    {"F": "F3", "명칭": "현장경제 경로", "경로": "Fan → Event → Local", "대표신호": "콘서트·투어·관객·숙박·교통"},
    {"F": "F4", "명칭": "산업전이 경로", "경로": "Fan → Brand/Industry", "대표신호": "광고·브랜드·앰버서더"},
    {"F": "F5", "명칭": "대중·글로벌 확산 경로", "경로": "Fan → Media → Mass/Global", "대표신호": "차트·방송·유튜브·해외활동"},
])
map_df = pd.DataFrame([{"raw factor 라벨(동결 LDA)": k, "F": v["f_code"], "명칭": v["f_name"], "경로": v["f_path"]} for k, v in pathway["mapping"].items()]).sort_values("F")
print("QA:", pathway["qa"])
print("전략문서 F명칭 == 매핑 파일 F명칭:", dict(zip(F_IMPACT_PATHWAY["F"], F_IMPACT_PATHWAY["명칭"])) == dict(zip(map_df["F"], map_df["명칭"])))
map_df.reset_index(drop=True)

QA: {'n_topics': 10, 'n_mapped_topics': 10, 'unmapped_topic_ratio': 0.0}
전략문서 F명칭 == 매핑 파일 F명칭: True


,raw factor 라벨(동결 LDA),F,명칭,경로
0,결속형(팬클럽·기부·커뮤니티),F1,팬덤결속 경로,Fan → Fan
1,소비력형(초동·판매·앨범),F2,직접소비 경로,Fan → Market
2,현장경제형(콘서트·투어·매진),F3,현장경제 경로,Fan → Event → Local
3,미디어노출형(방송·조회수),F4,산업전이 경로,Fan → Brand/Industry
4,차트·확산형(1위·빌보드·기록),F5,대중·글로벌 확산 경로,Fan → Media → Mass/Global


## 2. K→F 매핑 현황 — 동결 진단(K=10) 그대로

In [3]:
label_to_f = {k: v["f_code"] for k, v in pathway["mapping"].items()}
rows = []
for t, fid in diag["topic_to_factor"].items():
    lab = diag["factor_labels"][str(fid)]
    rows.append({"topic_id": f"K{int(t)}", "topic_keywords": "·".join(diag["topics_top_words"][t][:6]), "raw factor": lab, "F": label_to_f[lab]})
mapping_df = pd.DataFrame(rows).sort_values("topic_id").reset_index(drop=True)
n_topics, n_mapped = len(diag["topics_top_words"]), len(diag["topic_to_factor"])
print(f"QA: 매핑률 = {n_mapped}/{n_topics} ({n_mapped / n_topics:.0%}), 미매핑 Topic = {n_topics - n_mapped}개")
print("F별 구성 토픽 수:", mapping_df["F"].value_counts().sort_index().to_dict())
print("Topic Card 이름:", {c["topic_id"]: c["topic_name"] for c in cards})
print("덴드로그램 컷 높이:", round(space["dendro"]["cut_height"], 4), "| PCA 분산비:", [round(v, 3) for v in space["pca"]["var_ratio"]])
mapping_df

QA: 매핑률 = 10/10 (100%), 미매핑 Topic = 0개
F별 구성 토픽 수: {'F1': 1, 'F2': 2, 'F3': 3, 'F4': 2, 'F5': 2}
Topic Card 이름: {'K0': '음원차트기록형(기록·1위·차트·최초)', 'K1': '동남아현지보도형(보도·매체·인도네시아·기사)', 'K2': '예능방송출연형(예능·출연·mbc·sbs)', 'K3': '일본오리콘앨범형(일본·빌보드·오리콘·판매)', 'K4': '글로벌음반판매형(million·album·copies·chart)', 'K5': '팬클럽공식기부형(공식·팬클럽·기부·콘텐츠)', 'K6': '단독콘서트월드투어형(콘서트·투어·단독·월드투어)', 'K7': '브랜드앰버서더형(브랜드·광고·앰버서더·매진)', 'K8': '월드투어매진형(tour·concert·sold·world)', 'K9': '영화드라마출연형(드라마·ost·영화·출연)'}
덴드로그램 컷 높이: 0.7736 | PCA 분산비: [0.408, 0.308]


,topic_id,topic_keywords,raw factor,F
0,K0,기록·1위·발매·차트·데뷔·최초,차트·확산형(1위·빌보드·기록),F5
1,K1,무대·2026년·보도·8월·기사·인도네시아,현장경제형(콘서트·투어·매진),F3
2,K2,출연·예능·mbc·sbs·mc·kbs,미디어노출형(방송·조회수),F4
3,K3,앨범·1위·판매·이상·데뷔·일본,차트·확산형(1위·빌보드·기록),F5
4,K4,million·album·japan·chart·music·copies,소비력형(초동·판매·앨범),F2
5,K5,공식·팬클럽·유튜브·채널·팬덤·활동,결속형(팬클럽·기부·커뮤니티),F1
6,K6,콘서트·공연·보도·투어·단독·데뷔,현장경제형(콘서트·투어·매진),F3
7,K7,브랜드·모델·광고·앰버서더·콘서트·발탁,현장경제형(콘서트·투어·매진),F3
8,K8,tour·concert·fan·sold·world·seoul,소비력형(초동·판매·앨범),F2
9,K9,드라마·ost·출연·예능·영화·참여,미디어노출형(방송·조회수),F4


## 3. Topic Card — 최종 `topic_cards_v7.json` 포맷 확인 (문장이 가장 많이 배정된 카드 1개 출력)

In [4]:
def card_summary(c):
    return {"topic_id": c["topic_id"], "topic_name": c["topic_name"], "top_keywords": c["top_keywords"][:10],
            "connected": (c.get("connected_factor_label"), c.get("connected_f_pathway")),
            "대표_근거문장(최대3)": [b["text"][:80] + "…" for b in c["representative_bullets"][:3]],
            "대표_팬덤_Top5": {f["fandom"]: f["avg_topic_weight"] for f in c["representative_fandoms_top5"]}}
import pprint
print("카드 수:", len(cards), "| 카드별 대표 팬덤 수:", {c["topic_id"]: len(c["representative_fandoms_top5"]) for c in cards})
pprint.pprint(card_summary(cards[0]), width=140)

카드 수: 10 | 카드별 대표 팬덤 수: {'K0': 5, 'K1': 5, 'K2': 5, 'K3': 5, 'K4': 5, 'K5': 5, 'K6': 5, 'K7': 5, 'K8': 5, 'K9': 5}
{'connected': ('차트·확산형(1위·빌보드·기록)', 'F5 대중·글로벌 확산 경로'),
 'top_keywords': ['기록', '1위', '발매', '차트', '데뷔', '최초', 'k팝', '그룹', '누적', '앨범'],
 'topic_id': 'K0',
 'topic_name': '음원차트기록형(기록·1위·차트·최초)',
 '대표_근거문장(최대3)': ["뜬금없이 '잇츠 미' 등장..아일릿, 패러디 화제 속 美 '빌보드 200' 26위. 걸 그룹 아일릿(ILLIT)이 커리어 하이를 달성했다. 12…",
                  "ITZY의 데뷔곡 'Dalla Dalla' 뮤직비디오는 공개 24시간 만에 1710만 회 조회수를 기록해 K팝 데뷔 뮤직비디오 최고 기록을 세웠…",
                  "'Fly' 활동 당시 갓세븐은 빌보드 아티스트 100 차트에 45위로 데뷔해, 싸이가 2015년 세운 88위 기록을 넘어서며 역대 두 번째로 이…"],
 '대표_팬덤_Top5': {'CORTIS': 0.3312, 'GOT7': 0.2591, '빅마마': 0.222, '아일릿': 0.3051, '지코': 0.2617}}


## 4. Factor-specific Impact (전략문서 7절: share × loyalty / share × spillover) — `fan_persona_v7.json` 값 재현

In [5]:
mism_l = mism_s = 0; rows = []
for p in persona["fandoms"]:
    d = scores_by[p["fandom"]]
    for lab, share in d["factor_share"].items():
        f = label_to_f[lab]
        fl, fs = round(share * d["loyalty_score"], 4), round(share * d["spillover_score"], 4)
        if abs(fl - p["factor_specific_loyalty"][f]) > 0.0015: mism_l += 1
        if abs(fs - p["factor_specific_spillover"][f]) > 0.0015: mism_s += 1
        rows.append({"fandom": p["fandom"], "F": f, "factor_share": round(share, 4), "factor_specific_loyalty": fl, "factor_specific_spillover": fs})
fi = pd.DataFrame(rows)
print(f"factor_specific_loyalty 불일치 (팬덤×F 셀): {mism_l}/{len(fi)} | factor_specific_spillover 불일치: {mism_s}/{len(fi)}")
for name in ["BTS", "임영웅", "리센느(RESCENE)"]:
    d = scores_by[name]; p = next(x for x in persona["fandoms"] if x["fandom"] == name)
    print(f"\n=== {name} === loyalty={d['loyalty_score']}, spillover={d['spillover_score']}, dominant={d['dominant_factor']} → persona {p['persona']} "
          f"(Top-2: {p['top2_factors'][0]['f_code']} {p['top2_factors'][0]['share']}, {p['top2_factors'][1]['f_code']} {p['top2_factors'][1]['share']})")
    print(fi[fi["fandom"] == name].sort_values("factor_share", ascending=False).head(3).to_string(index=False))

factor_specific_loyalty 불일치 (팬덤×F 셀): 0/500 | factor_specific_spillover 불일치: 0/500

=== BTS === loyalty=0.931, spillover=1.0, dominant=현장경제형(콘서트·투어·매진) → persona 글로벌투어형 (Top-2: F3 0.5119, F5 0.2008)
fandom  F  factor_share  factor_specific_loyalty  factor_specific_spillover
   BTS F3        0.5119                   0.4766                     0.5119
   BTS F5        0.2008                   0.1869                     0.2008
   BTS F2        0.1455                   0.1355                     0.1455

=== 임영웅 === loyalty=0.899, spillover=0.702, dominant=현장경제형(콘서트·투어·매진) → persona 집단동원형 (Top-2: F3 0.3627, F1 0.2017)
fandom  F  factor_share  factor_specific_loyalty  factor_specific_spillover
   임영웅 F3        0.3627                   0.3261                     0.2546
   임영웅 F1        0.2017                   0.1813                     0.1416
   임영웅 F5        0.1673                   0.1504                     0.1174

=== 리센느(RESCENE) === loyalty=0.444, spillover=0.304, dominant=현장경제형(콘서트·투어·

## 5. Fan Persona(Top-2 Factor 조합) — persona_counts 43/31/17/9 재현

In [6]:
table = persona["persona_table_definition"]
combo_counts = {}; name_mis = 0
for p in persona["fandoms"]:
    d = scores_by[p["fandom"]]
    top2 = sorted(sorted(d["factor_share"].items(), key=lambda kv: -kv[1])[:2], key=lambda kv: label_to_f[kv[0]])
    key = "|".join(label_to_f[k] for k, _ in top2)
    if table[key] != p["persona"]: name_mis += 1
    combo_counts[table[key]] = combo_counts.get(table[key], 0) + 1
print(f"페르소나 명칭 재계산 불일치: {name_mis}/{len(persona['fandoms'])}")
print("재계산 persona_counts:", dict(sorted(combo_counts.items(), key=lambda x: -x[1])), "| JSON:", persona["persona_counts"], "| PCA JSON:", space["pca"]["persona_counts"])
print(f"이론상 조합 수 (M=5, 2개 조합) = {5 * 4 // 2}개, 실제 실현 = {len(combo_counts)}개")
pd.DataFrame([{"top2_factor_combo": k, "persona": v, "n_fandoms": combo_counts.get(v, 0)} for k, v in table.items()]).sort_values("n_fandoms", ascending=False).reset_index(drop=True)

페르소나 명칭 재계산 불일치: 0/100
재계산 persona_counts: {'글로벌투어형': 43, '현장상업형': 31, '원정소비형': 17, '집단동원형': 9} | JSON: {'글로벌투어형': 43, '집단동원형': 9, '원정소비형': 17, '현장상업형': 31} | PCA JSON: {'글로벌투어형': 43, '집단동원형': 9, '원정소비형': 17, '현장상업형': 31}
이론상 조합 수 (M=5, 2개 조합) = 10개, 실제 실현 = 4개


,top2_factor_combo,persona,n_fandoms
0,F3|F5,글로벌투어형,43
1,F3|F4,현장상업형,31
2,F2|F3,원정소비형,17
3,F1|F3,집단동원형,9
4,F1|F2,핵심소비형,0
5,F1|F4,브랜드동반형,0
6,F2|F4,소비상업형,0
7,F1|F5,글로벌결속형,0
8,F2|F5,글로벌소비형,0
9,F4|F5,산업확장형,0


## 6. 실루엣 게이트 — 라이브 10,020건 재적합이 동결 스냅샷을 대체하지 못한 이유 (진단 대조)

In [7]:
grid = pd.DataFrame(diag["k_grid"]).assign(snapshot="동결 7,350").merge(pd.DataFrame(diag_live["k_grid"]).assign(snapshot="라이브 10,020"), how="outer")
print("라이브 K→factor:", diag_live["topic_to_factor"], "| 라이브 factor 라벨:", diag_live["factor_labels"])
print(f"게이트: 동결 실루엣 {diag['meta_factor_silhouette']} vs 라이브 {diag_live['meta_factor_silhouette']} → 라이브 재적합은 보고서 본문 지표로 채택되지 않음.")
grid.pivot(index="k", columns="snapshot", values=["perplexity", "coherence", "diversity", "stability", "composite_rank_sum"])

라이브 K→factor: {'0': 3, '1': 4, '2': 0, '3': 1, '4': 0, '5': 0, '6': 0, '7': 2} | 라이브 factor 라벨: {'0': '현장경제형(콘서트·투어·매진)', '1': '소비력형(초동·판매·앨범)', '2': '결속형(팬클럽·기부·커뮤니티)', '3': '브랜드·상업형(광고·앰버서더)', '4': '차트·확산형(1위·빌보드·기록)'}
게이트: 동결 실루엣 0.267 vs 라이브 0.046 → 라이브 재적합은 보고서 본문 지표로 채택되지 않음.


perplexity            coherence            diversity            stability            composite_rank_sum           
snapshot   동결 7,350 라이브 10,020  동결 7,350 라이브 10,020  동결 7,350 라이브 10,020  동결 7,350 라이브 10,020           동결 7,350 라이브 10,020
k                                                                                                                          
8            3046.2     3468.8    -2.398     -2.390     0.900      0.875     0.420      0.421                9.0        5.0
10           3108.2     3613.8    -2.348     -2.512     0.850      0.780     0.460      0.421                8.0       15.0
12           3134.4     3679.9    -2.361     -2.503     0.842      0.783     0.371      0.370               14.0       15.0
15           3255.2     3723.9    -2.425     -2.660     0.780      0.787     0.415      0.362               17.0       18.0
20           3423.6     3828.5    -2.312     -2.486     0.750      0.720     0.352      0.347               19.0       20.0
25           3517.0     3885.6    -2.480     -2.347     0.768      0.796     0.317      0.333               24.0       16.0
30           3636.8     4065.0    -2.245     -2.542     0.767      0.787     0.288      0.344               21.0       23.0

## 7. QA 요약 (전략문서 11절 체크리스트)

| 항목 | 결과 |
|---|---|
| K→F 매핑률 | 10/10 (미매핑 0) — `factor_pathway_map_v7.json` qa |
| Factor-specific Impact 재현 | 4절: 불일치 0셀이면 `fan_persona_v7.json` 값이 동결 점수에서 그대로 재계산됨 |
| 페르소나 분포 | 5절: 글로벌투어형 43 · 현장상업형 31 · 원정소비형 17 · 집단동원형 9 (10개 조합 중 4개 실현) |
| 실루엣 게이트 | 동결 0.267 채택, 라이브 0.046 기각 |

## 최종 노트 — r22 노트북과 다른 지점

- r22: K=8/M=6/0.154, "브랜드·상업형"·"미디어노출형" 두 라벨 모두 존재. 최종 동결: K=10/M=5/0.267, F4 산업전이 경로에 raw 라벨 "미디어노출형(방송·조회수)"이 매핑됨(`factor_pathway_map_v7.json` rationale 참고) — 라벨 자동 부여 규칙의 결과이며 광고 신호는 별도 키워드 지수(`ad_commercial_index_v7.json`)로 보완됐다.
- 라이브 재적합은 K-grid 승자가 K=8로 바뀌고 실루엣이 0.046으로 떨어져 게이트를 넘지 못했다 — 최종 보고서는 loyalty/spillover만 라이브로 쓴다.